In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/NYTimes/code/Initialization/Initialization.ipynb"

In [ ]:
# Import packages
import os
import pickle

import numpy as np 
import pandas as pd 

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from scipy import stats
import scipy.optimize as optimize 

In [ ]:
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

In [ ]:
org_data = pd.DataFrame()#pd.read_csv("/Users/Farid Srouji/Documents/Research/master/globaldata/options_VIX.csv")

from yahoo_fin.stock_info import get_data

# VIX= get_data("^VIX", start_date="05/31/2020", end_date="10/08/2021", index_as_date = True, interval="1d")
VIX=df_vix#pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/vix.pkl', compression=None)

# SP500= get_data("^GSPC", start_date="05/31/2020", end_date="10/08/2021", index_as_date = True, interval="1d")
SP500=df_sp500#pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/sp500.pkl', compression=None)


VIX["Date"]=VIX.index
VIX["Return"]=VIX["VIX return"]

SP500["Date"]=SP500.index
SP500["Return"]=SP500["S&P500 return"]

VIX = VIX.loc[:,['Date','Return']]
SP500 = SP500.loc[:,['Date','Return']]


In [ ]:
# Convert date data
VIX['Date'] = pd.to_datetime(VIX['Date'])
org_data["Date"] = VIX['Date']
org_data["VIX return"] = VIX["Return"]
org_data["S&P500 return"] = SP500["Return"]

In [ ]:
org_data = org_data.ffill()

In [ ]:
pivot = ['Date','VIX return', 'S&P500 return']

# Save the original data
full_data = org_data.copy()

# Shift the pivot data
full_data[pivot] = full_data[pivot].shift(periods=-1)

# Drop the last row
full_data = full_data.drop(full_data.index[-1])

In [ ]:
# Format foreign volatility indices as input data

def correspond_foreign_vol(date,data):
    """
    find the 'Price' of the date that is most recent to the given 'Date' in the 'data'
    """
    
    while True:
        date = date - pd.Timedelta('1 Day')  # go back one day 
        result_series = data['Return'].loc[data['Date']==date]  # find the 'Price' in the data that matches with the 'date'
        
        if not result_series.empty:  # if not empty (which means there is a row that matches with the 'date')
            result_series = result_series.reset_index()  # reset index
            result_value = result_series['Return'][0]  # and get the value
            return result_value

# Apply function

full_data['VIX return'] = full_data['Date'].apply(correspond_foreign_vol,data=VIX)


In [ ]:
# Calculate return

SP500_yesterday = org_data['S&P500 return'].shift(periods=1)  # Series made up of shifted VIX
return_array = (org_data['S&P500 return']-SP500_yesterday) / SP500_yesterday  # Calculate return

In [ ]:
# GARCH model
def garch_forward(return_rate,variance,coefficients):
    ''' data type: float, float, 1d array(length=3)'''
    
    # Coefficients
    alpha,beta,omega = coefficients
    # Calculate
    return omega + alpha*return_rate*return_rate + beta*variance

In [ ]:
full_data['VIX return']

In [ ]:
# Set VIX as an initial variance in the model

initial_vix_ln_return= full_data['VIX return'][1] 
initial_variance = initial_vix_ln_return*initial_vix_ln_return/2520000 

In [ ]:
initial_vix_ln_return

In [ ]:
# Function for optimization

def garch_for_optimization(array): 
    ''' data type: 1d array(length=3)'''
    
    # Coeffcients
    alpha,beta,omega = array 
    
    # Variables
    sum_probability = 0 # to maximize
    variance = initial_variance
    
    for i in range(1,return_array_train.shape[0]):  # exclude the first value because it's nan.
        return_rate = return_array_train[i]  
        
        # in case something goes wrong
        if variance<=0:
            print("Negative variance")
            break
        
        # calculate probability in a single day
        probability = -np.log(variance) - return_rate * return_rate / variance
        
        # add to the sum
        sum_probability += probability
        
        # calculate next day's variance by GARCH
        variance = garch_forward(return_rate,variance,array)
   
    return -sum_probability # note the sign(-); because scipy.optimize requires a function to be minimized

In [ ]:
# Optimize
# if execute is set TRUE, optimization which takes some time will be initiated
execute = True    
return_array_train=return_array
if execute:
    bounds = optimize.Bounds([0,0,0],[1,1,np.inf])
    initial_guess = [0.14,0.76,2.97]  # V-Lab's estimate

    # Trust-constr performed the best 
    optimize_res_trust = optimize.minimize(garch_for_optimization,initial_guess,method='trust-constr',bounds=bounds)

    # # Pickle the result since it takes too much time
    with open('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/optimize_res_trust_vix_ln_return.pkl','wb') as file:
        pickle.dump(optimize_res_trust,file)

In [ ]:
optimize_res_trust

In [ ]:
# Load the pickle: optimize_res_trust
with open('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/optimize_res_trust_vix_ln_return.pkl','rb') as file:
    optimize_res_trust = pickle.load(file)

# Optimization result
print('optimization result:')
print('alpha =',optimize_res_trust.x[0])
print('beta =',optimize_res_trust.x[1])
print('omega =',optimize_res_trust.x[2])

In [ ]:
return_array

In [ ]:
optimize_res_trust.x

In [ ]:
# Estimate historical volatility with estimated coefficients

variance_array = np.zeros(full_data.shape[0],)  # Create array to store variance

# Values to be pre-assigned
variance_array[0] = np.nan  # Historical volatility cannot be estimated for the first one in full_data (06/03/2009)
variance_array[1] = initial_variance 

# Calculate historical volatilities using GARCH
for i in range(2,full_data.shape[0]):
    variance_array[i]=garch_forward(return_array[i-1],variance_array[i-1],optimize_res_trust.x)
    
# Adjust value to compare with VIX (elaborated in 2-3-2)  
historical = variance_array#np.sqrt(variance_array * 252) * 100

# Add to the dataset
full_data['Historical Volatility Ln Return'] = historical
full_data = full_data.dropna()  # Drop NA (because of nan in historical volatility)

In [ ]:
pd.DataFrame(data=variance_array)

In [ ]:
full_data.head(40)

In [ ]:
# Before visualizing, few matplotlib settings
mpl.rcParams['axes.labelsize'] = 'x-large'
mpl.rcParams['axes.labelpad'] = 5.5  # space between axis and label
mpl.rcParams['axes.titlesize'] = 'large'
mpl.rcParams['axes.titlepad'] = '20.0'

mpl.rcParams['legend.fontsize'] = 'x-large'

# mpl color settings
default_clrs = plt.rcParams['axes.prop_cycle'].by_key()['color']

# and seaborn settings
sns.set_style('darkgrid')

In [ ]:
# Compare with VIX (Visualization)
pd.plotting.register_matplotlib_converters()  # because of compatilbility issue with pd.Timestamp and matplotlib

fig, ax = plt.figure(figsize=(20,7)), plt.axes()
sns.lineplot(data=full_data,x='Date',y='VIX return',label='VIX Ln Return(target)',ax=ax)
sns.lineplot(data=full_data,x='Date',y='Historical Volatility Ln Return',label='Historical Volatility Ln Return (prediction)',ax=ax)
plt.ylabel('Volatility Ln Return')
plt.show()

fig.savefig("/Users/Farid Srouji/Documents/Research/master/NYTimes/images/PredictionModels/HistoricalVolatilityLnReturn.png", bbox_inches='tight')


In [ ]:
# Calculate error
abs_garch = abs(full_data['VIX return']-full_data['Historical Volatility Ln Return'])
square_garch = (full_data['VIX return']-full_data['Historical Volatility Ln Return'])**2

print('Total set')
print('MAE:',abs_garch.mean())
print('RMSE:',square_garch.mean()**0.5)

print('\nTest set ([-516:])')
print('MAE:',abs_garch[-516:].mean())
print('RMSE:',square_garch[-516:].mean()**0.5)

In [ ]:
full_data

In [ ]:
df_part3["Date"] = df_part3.index
df_part3.reset_index(drop=True,inplace=True)
df_part3.drop(columns=["vix_ln_return", "sp500_ln_return"], inplace=True)

In [ ]:
full_data = pd.merge(full_data,df_part3,how='outer', on='Date')

In [ ]:
full_data = full_data.ffill()

In [ ]:
full_data = full_data.sort_values(by="Date")
full_data.index = full_data.Date

In [ ]:
# Split data into train data(including validation data) and test data
split_ratio = 0.7

data = full_data.set_index('Date')
len_data = full_data.shape[0]

train_org = full_data.iloc[:int(len_data*split_ratio),]
test_org = full_data.iloc[int(len_data*split_ratio):,]

print('train set',train_org.shape)
print('test set',test_org.shape)